In [0]:
%pip install azure-storage-file-datalake azure-identity pandas pyarrow


In [0]:
dbutils.library.restartPython()

In [0]:
import azure.identity
import azure.storage.filedatalake

print("O SDK da Azure foi encontrado e carregado com sucesso!")

In [0]:
from dotenv import load_dotenv
import os

load_dotenv("/Workspace/Users/clebercampos7@hotmail.com/estagio-empregadados-turma-2/.env")

client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
storage_account = os.getenv("ADLS_STORAGE_ACCOUNT")

print("client_id carregado:", client_id is not None)
print("tenant_id carregado:", tenant_id is not None)
print("client_secret carregado:", client_secret is not None)
print("storage_account carregado:", storage_account is not None)

In [0]:
# Configurações OAuth do Service Principal passadas a cada chamada do Spark
# Isso garante que a autenticação ocorra diretamente nos executores do cluster
adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

print("Dicionário adls_options preparado para injeção no spark.read.")

In [0]:
# 1. Definindo o caminho com curingas para a tabela de clientes
container_name = "raw"
tabela_alvo = "ecommerce_clientes"

# O caminho navega pelas pastas de Ano/Mes/Dia/Hora usando /*/*/*/*/
file_path = f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/real-time-data/*/*/*/*/{tabela_alvo}.parquet"

print(f"Iniciando a leitura dos arquivos em:\n{file_path}\n")

try:
    # 2. Lendo os dados e injetando as credenciais com **adls_options
    df_clientes = spark.read.options(**adls_options).format("parquet").load(file_path)

    print("✅ Conexão e leitura concluídas com sucesso! Exibindo os dados:")
    display(df_clientes)

except Exception as e:
    print(f"❌ Erro durante a leitura:\n{e}")